# Simulation Diagnostics

A simulation can run to the end without any error and still give inaccurate results, if its time step is too large compared to the physical processes it solves. In this tutorial, we will learn to use the **diagnostics** of SAETASS, which analyze a configured simulation *before running it* and warn about time steps that compromise its accuracy.

The diagnostics rely on **dimensionless numbers**, such as the Courant number of advection, which compare the time step and the grid spacing to the physics of each operator. As we saw in the first tutorial, SAETASS converts every input to its canonical units, but dimensionless numbers do not depend on units at all: they characterize the discrete problem itself.

## 1. A simulation to diagnose

We start from the advection-diffusion problem with a central source of the fourth tutorial: a stellar wind carries particles outwards while they diffuse, during 2 Myr.

In [1]:
import math
import warnings

import astropy.units as u
import numpy as np

from saetass import units as su
from saetass.diagnostics import WARNING_SPLITTING_RATIO
from saetass.grid import Grid
from saetass.solver import Solver
from saetass.state import State
from saetass.utils.energy_losses import Particle


def make_solver(t_max, num_timesteps, D_values, source=None):
    grid = Grid.uniform(
        r_min=0.0 * u.pc,
        r_max=20.0 * u.pc,
        num_r_cells=200,
        t_min=0.0 * u.Myr,
        t_max=t_max,
        num_timesteps=num_timesteps,
    )
    r = grid.r_centers
    operator_params = {
        "advection": {
            "v_centers": np.full(grid.num_cells_r, 5.0) * u.km / u.s,
            "limiter": "vanleer",
            "order": 2,
            "cfl": 0.5,
            "inflow_value_U": 0.0 * su.AREA * su.PSI_P,
        },
        "diffusion": {"D_values": D_values},
        "source": {
            "source": source
            if source is not None
            else 100.0 * np.exp(-(r**2) / (2 * (0.5 * u.pc) ** 2)) * su.SOURCE_PSI_P
        },
    }
    state = State(grid, psi_p=np.zeros(grid.shape) * su.PSI_P, particle=Particle.PROTON)
    return Solver(
        grid=grid,
        state=state,
        problem_type="advection-diffusion-source",
        operator_params=operator_params,
    )


D_values = np.full(200, 0.5) * u.pc**2 / u.Myr
solver = make_solver(2.0 * u.Myr, 200, D_values)

╭─ SAETASS: Solver for Astroparticle Equation of Transport Analysis in Spherical Symmetry ─╮              
              │                                                                                          │              
              │                                     :::::::::::::::                                      │              
              │                                 :::::::::::::::::::::::                                  │              
              │                               :::::::::++++++++++::::::::                                │              
              │                              ::::::::++++++++++++++::::::::                              │              
              │                             :::::::++++++++++++++++++:::::::                             │              
              │                            ::::::::++++++++++++++++++::::::::                            │              
              │                            :::::::++++++++++++++++++++:::::::                            │              
              │                            :::::::+++++++++++++++++++::::::::                            │              
              │                            ::::::::++++++++++++++++++::::::::                            │              
              │                             ::::::::::++++++++++++++                                     │              
              │                              :::::::::::::::++++++                                       │              
              │                                :::::::::::::::::::::                                     │              
              │                                   ::::::::::::::::::::::                                 │              
              │                                       ++:::::::::::::::::::                              │              
              │                                     ++++++++++++::::::::::::                             │              
              │                                   ++++++++++++++++++:::::::::                            │              
              │                           ::::::::++++++++++++++++++++::::::::                           │              
              │                           :::::::++++++++++++++++++++++:::::::                           │              
              │                           :::::::++++++++++++++++++++++:::::::                           │              
              │                            :::::::++++++++++++++++++++:::::[

## 2. Running the diagnostics

The diagnostics of a configured `Solver` are obtained with its `diagnostics()` method. It does not advance the simulation, so it can be called before `run()` or `step()`. Printing the result gives a summary report:

In [2]:
diagnostics = solver.diagnostics()
print(diagnostics)

SAETASS diagnostics: time step 0.01 Myr, worst case over 3 sampled times
  advection  step 0.005 Myr, timescale 3.931 Myr, courant_number = 0.2544, cfl_subcycles = 1
  diffusion  step 0.005 Myr, timescale 808.1 Myr, fourier_number = 0.3713
  source     step 0.01 Myr, timescale -
  splitting_ratio = 0.002544, cell_peclet = 1.028


## 3. Reading the report

For each operator, the report shows:

- The **time step** it integrates in each call. With the default Strang splitting, all operators but the last one are applied twice per step, over half a time step each, which is why advection and diffusion integrate 0.005 Myr while the source integrates the full 0.01 Myr.
- Its **characteristic timescale**, the time over which it changes the solution significantly: the time to cross the domain at the wind speed for advection, $L / \max|v|$, and the diffusion time across the domain for diffusion, $L^2 / \max D$. Constant sources have none, as we will see in section 7.
- Its **dimensionless numbers**:
  - The **Courant number** of advection, $\max|v|\,\Delta t / \Delta r$, is the number of cells the wind crosses in one step. The advection solver sub-cycles automatically to keep it stable, so `cfl_subcycles` tells how many sub-steps that costs per call.
  - The **Fourier number** of diffusion, $D\,\Delta t / \Delta r^2$ on uniform grids, measures how far particles diffuse in one step in units of the cell size.

Below the operators, the report shows the numbers that involve several of them:

- The **splitting ratio** $\Delta t / \tau_\mathrm{min}$ compares the time step to the shortest operator timescale. Operator splitting assumes that the operators barely change the solution within a step, so this ratio must stay well below one: the diagnostics warn when it exceeds `WARNING_SPLITTING_RATIO`, which is 0.1.
- The **cell Péclet number** $|v|\,\Delta r / D$ tells which of advection (above one) or diffusion (below one) dominates at the scale of a cell. It is the first of the *cell numbers*, which compare two operators at the scale of a cell; we will meet the others in section 6.

Every value is also available programmatically, with timescales as Quantities that can be converted to any unit:

In [3]:
advection = diagnostics["advection"]
print("Advection crossing time:", advection.timescale.to(u.kyr))
print("Courant number:", advection.numbers["courant_number"])
print("Splitting ratio:", diagnostics.numbers["splitting_ratio"])

Advection crossing time: 3930.8230017318156 kyr
Courant number: 0.2543996510551212
Splitting ratio: 0.0025439965105511684


Here every number is small: this time step is well suited to the problem.

## 4. When the time step is too large

Let us now simulate 10 Myr with only two steps of 5 Myr each, longer than the time it takes the wind to cross the domain. The diagnostics issue a `DiagnosticsWarning`, which explains the problem and suggests a number of timesteps:

In [4]:
coarse = make_solver(10.0 * u.Myr, 2, D_values)
coarse_diagnostics = coarse.diagnostics()

/var/folders/hk/9yxq4rz17sx3kwjfrr5sv9b00000gn/T/ipykernel_21487/2521860412.py:2: DiagnosticsWarning: The time step of 5 Myr is 1.272 times the shortest operator timescale (advection, 3.931 Myr): splitting and time-integration errors may be significant. Use at least 26 timesteps (currently 2).
  coarse_diagnostics = coarse.diagnostics()


Since the splitting ratio grows linearly with the time step, the suggested number of timesteps brings it down to the warning threshold. Following the suggestion, the warning disappears:

In [5]:
ratio = coarse_diagnostics.numbers["splitting_ratio"]
suggested = math.ceil(2 * ratio / WARNING_SPLITTING_RATIO)
print(f"Splitting ratio {ratio:.2f}: using {suggested} timesteps")

with warnings.catch_warnings():
    warnings.simplefilter("error")  # any warning would raise an exception here
    refined_diagnostics = make_solver(10.0 * u.Myr, suggested, D_values).diagnostics()

print("New splitting ratio:", refined_diagnostics.numbers["splitting_ratio"])

Splitting ratio 1.27: using 26 timesteps
New splitting ratio: 0.09784601963658342


Warnings can also be disabled with `diagnostics(warn=False)`, and the descriptions of the problems found are always available in `diagnostics.warnings`.

## 5. Time-dependent parameters

Parameters that depend on time change their dimensionless numbers during the simulation. The diagnostics therefore evaluate them at several times, evenly spread over the time grid (three by default, set with `n_samples`), and report the **worst case**. Let us make the diffusion coefficient grow tenfold during the simulation:

In [6]:
def growing_D(t):
    return np.full(200, 0.5) * (1.0 + 9.0 * t / (2.0 * u.Myr)) * u.pc**2 / u.Myr


dynamic = make_solver(2.0 * u.Myr, 200, growing_D).diagnostics(n_samples=5)
print("Sampled times:", dynamic.sampled_times)
print("Fourier number, constant D:", diagnostics["diffusion"].numbers["fourier_number"])
print("Fourier number, growing D: ", dynamic["diffusion"].numbers["fourier_number"])

Sampled times: [0.  0.5 1.  1.5 2. ] Myr
Fourier number, constant D: 0.37125937499999995
Fourier number, growing D:  3.71259375


The reported Fourier number is the one at the end of the simulation, when the diffusion coefficient is largest.

### A note on the Fourier number

Diffusion is integrated with the Crank-Nicolson scheme, which is stable for any time step and guarantees non-negative distributions when the Fourier number is at most one. Larger values do not make it unstable, but they reduce its accuracy on features that span only a few cells, such as sharp source profiles. Smooth solutions tolerate Fourier numbers of hundreds, which are common in astrophysical setups, so the diagnostics report this number without warning about it: when a simulation has sharp features, it is worth checking that its Fourier number stays moderate.

## 6. Energy losses and the spatial grid

Energy losses add a new question: do particles travel far enough to be resolved by the spatial grid before losing their energy? Comparing the time to cross a cell with the **loss time** $p / |\dot{p}|$ gives the **cell Damköhler numbers**, one for each transport process:

$$
\mathrm{Da}_v = \frac{\Delta r / |v|}{p / |\dot{p}|}, \qquad \mathrm{Da}_D = \frac{\Delta r^2 / D}{p / |\dot{p}|}.
$$

When a Damköhler number is above one, particles lose their energy before leaving the cell where they are: their spatial profile at that momentum is narrower than a cell, so the grid cannot resolve it.

Let us add a momentum axis to our problem, from 1 GeV/c to 10 TeV/c, with losses $\dot{p} = -b\,p^2$, whose loss time $1 / (b\,p)$ decreases with momentum. We only need the diagnostics, so we do not run the simulation:

In [7]:
p = np.logspace(0, 4, 60) * su.MOMENTUM
r = np.linspace(0.0, 20.0, 200) * u.pc
shape = (p.size, r.size)
b = 0.01 / (su.MOMENTUM * u.Myr)
D = 0.5 * u.pc**2 / u.Myr

loss_grid = Grid(r_centers=r, p_centers=p, t_grid=np.linspace(0.0, 1.0, 2001) * u.Myr)
loss_solver = Solver(
    grid=loss_grid,
    state=State(
        loss_grid, psi_p=np.zeros(shape) * su.PSI_P, particle=Particle.ELECTRON
    ),
    problem_type="advection-diffusion-loss",
    operator_params={
        "advection": {
            "v_centers": np.full(shape, 5.0) * u.km / u.s,
            "limiter": "vanleer",
            "order": 2,
            "cfl": 0.5,
            "inflow_value_U": 0.0 * su.AREA * su.PSI_P,
        },
        "diffusion": {"D_values": np.full(shape, 1.0) * D},
        "loss": {
            "P_dot": -b * p[:, None] ** 2 * np.ones(shape),
            "limiter": "vanleer",
            "order": 2,
            "cfl": 0.5,
            "inflow_value_psi": 0.0 * su.PSI_P,
        },
    },
)
print(loss_solver.diagnostics())

SAETASS diagnostics: time step 0.0005 Myr, worst case over 3 sampled times
  advection  step 0.00025 Myr, timescale 3.931 Myr, courant_number = 0.01272, cfl_subcycles = 1
  diffusion  step 0.00025 Myr, timescale 808.1 Myr, fourier_number = 0.01856
  loss       step 0.0005 Myr, timescale 0.01 Myr, courant_number = 0.3453, cfl_subcycles = 1
  splitting_ratio = 0.05, cell_peclet = 1.028, cell_damkohler_advection = 1.965, cell_damkohler_diffusion = 2.02


Both Damköhler numbers are close to two: at the highest momenta, the grid does not resolve the spatial profile. Since the loss time decreases with momentum, the Damköhler numbers grow with it, and we can find the momentum above which the grid stops resolving the profile. For diffusion, $\mathrm{Da}_D = 1$ when $p = D / (b\,\Delta r^2)$:

In [8]:
dr = loss_grid.dr[0]
print("Diffusion resolved up to p =", (D / (b * dr**2)).to(su.MOMENTUM))

Diffusion resolved up to p = 4950.124999999999 GeV_c


Unlike the splitting ratio, these numbers do not depend on the time step: refining the time grid does not help, only refining the spatial grid does. They are reported without a warning, because particles with fast losses are often *meant* to stay confined near their sources, and the momenta where that happens may not matter for the problem at hand.

## 7. Time-dependent sources

A natural question is whether a source has a timescale of its own: injecting $1\,\psi_p$ per Myr is not the same as injecting it per year. The magnitude of the source, however, does not affect the accuracy of the simulation. The transport equation is linear, so the solution is the sum of the part coming from the initial distribution and the part coming from the source, and multiplying the source by any factor only multiplies the second part by the same factor, with the same relative errors.

What matters is how fast the source **changes**: each step injects the source evaluated at the beginning of the step, which is accurate only if it barely changes within the step. The diagnostics therefore report the variation timescale of time-dependent sources, $\tau_Q = \max|Q| / \max|\partial_t Q|$, and the relative change of the source over one step, $\Delta t / \tau_Q$, as `source_variation`. Let us make the source of our first problem oscillate with a period of 20 kyr:

In [9]:
def pulsed_source(amplitude, period):
    def source(r, p, t):
        profile = np.exp(-(r**2) / (2 * (0.5 * u.pc) ** 2))
        pulse = 1.0 + np.sin(2 * np.pi * (t / period).to_value(u.one))
        return amplitude * profile * pulse * su.SOURCE_PSI_P

    return source


pulsed = make_solver(2.0 * u.Myr, 200, D_values, pulsed_source(100.0, 20 * u.kyr))
pulsed_diagnostics = pulsed.diagnostics()

/var/folders/hk/9yxq4rz17sx3kwjfrr5sv9b00000gn/T/ipykernel_21487/2851250967.py:11: DiagnosticsWarning: The time step of 0.01 Myr is 1.414 times the shortest operator timescale (source, 0.007071 Myr): splitting and time-integration errors may be significant. Use at least 2829 timesteps (currently 200).
  pulsed_diagnostics = pulsed.diagnostics()


The source now changes faster than the time step, and its timescale becomes the shortest of the simulation, so the splitting ratio warns about it. Scaling the source by a million does not change its diagnostics at all:

In [10]:
huge = make_solver(2.0 * u.Myr, 200, D_values, pulsed_source(1e8, 20 * u.kyr))
for name, diag in [
    ("Q0 = 100", pulsed_diagnostics),
    ("Q0 = 1e8", huge.diagnostics(warn=False)),
]:
    source = diag["source"]
    print(
        f"{name}: timescale {source.timescale.to(u.kyr):.4g}, "
        f"source_variation {source.numbers['source_variation']:.4g}"
    )

Q0 = 100: timescale 7.071 kyr, source_variation 1.414
Q0 = 1e8: timescale 7.071 kyr, source_variation 1.414


## Summary

In this tutorial, you learned:
1. How to analyze a configured simulation with `solver.diagnostics()`, before running it.
2. How to read the **timescales** and **dimensionless numbers** of each operator: Courant, Fourier, splitting ratio and cell Péclet numbers.
3. How the diagnostics **warn** about time steps that are too large and suggest a number of timesteps.
4. How **time-dependent parameters** are evaluated at several times to report the worst case.
5. How the **cell Damköhler numbers** tell whether the spatial grid resolves particles with fast energy losses.
6. Why the timescale of a source is set by its **variation**, not by its magnitude.

Running the diagnostics before long simulations is a cheap way to make sure their time grid suits the physics they solve.